# encoder-decoder-symmetric — worked example 1: Three-Stage Encoder-Decoder: Shape Preserved End-to-End

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `encoder-decoder-symmetric`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A symmetric encoder-decoder downsamples with pooling or strided conv in the encoder, then precisely mirrors each stage with an upsample in the decoder. The encoder doubles channel count while halving spatial dimensions; the decoder reverses this. The key invariant is that the total spatial downsampling factor (product of all pool strides) must equal the total upsampling factor for the output to match the input shape.

## Worked solution

We build a three-stage autoencoder with channels 1 → 8 → 16 → 32 → 16 → 8 → 1 and spatial reduction 8× (three pool stages of stride 2).

**Encoder structure:** Three Conv2d + ReLU + MaxPool2d(2) blocks. Each MaxPool halves spatial dimensions. Channel trajectory: in_channels → 8 → 16 → 32.

**Decoder structure (mirror):** Three Upsample(2) + Conv2d + ReLU blocks. Each Upsample doubles spatial dimensions. Channel trajectory: 32 → 16 → 8 → in_channels. The final Conv2d does NOT use ReLU, so it can reconstruct values outside [0, 1] if needed.

**Why the shapes match:** 3 MaxPool2d(2) stages give total downsampling factor 8. 3 Upsample(2) stages give total upsampling factor 8. They cancel perfectly. For any input `(B, C, H, W)` where H and W are divisible by 8, the output is `(B, C, H, W)`.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(5)

class ThreeStageAutoencoder(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 8,  kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8,           16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,          32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(16, 8,  kernel_size=3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(8, in_channels, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = ThreeStageAutoencoder(in_channels=1)
model.eval()

for h in [8, 16, 32]:
    x = t.randn(2, 1, h, h)
    out = model(x)
    assert out.shape == x.shape, f"Shape mismatch at H={h}: {out.shape} != {x.shape}"
    print(f"Input (2,1,{h},{h}) -> Output {tuple(out.shape)} ✓")

print("All shapes preserved.")